# 1B — Checker Prompt Generation

**Purpose:** Generates token-without-concept prompts for the token independence check. No model loading, no activation extraction — only prompt generation and validation using the tokenizer.

In [ ]:
# Cell 1 – Configuration
import random
import numpy as np

N_TARGET = 50

LANG = "P"
MODEL = "QW"
PREFIX = f"{LANG}_{MODEL}_"
OVERSHOOT_FACTOR = 3
random.seed(42)
np.random.seed(42)

print(f"Target prompts per object: {N_TARGET}")
print(f"Overshoot factor: {OVERSHOOT_FACTOR}")

In [ ]:
# Cell 2 – Imports & environment
import subprocess, sys, os, shutil, ast as ast_module
for pkg in ["pandas", "pyarrow", "transformers"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

import pandas as pd

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    mp = "/content/drive"
    subprocess.run(["fusermount", "-uz", mp], capture_output=True)
    if os.path.isdir(mp):
        shutil.rmtree(mp, ignore_errors=True)
    drive.mount(mp)
    DATA_DIR = "/content/drive/MyDrive/DATA/New-Atlas"
else:
    DATA_DIR = "/Users/piotrwilam/Data/New-Atlas"

OUTPUT_FILE = f"{PREFIX}1B_checker_prompts.parquet"
os.makedirs(DATA_DIR, exist_ok=True)

print(f"Environment: {'Colab' if IN_COLAB else 'Local'}")
print(f"Data dir: {DATA_DIR}")

In [ ]:
# Cell 3 – Load tokenizer only (no model needed)
from transformers import AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-Coder-7B"

LANG = "P"
MODEL = "QW"
PREFIX = f"{LANG}_{MODEL}_"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f"Tokenizer loaded: vocab size = {tokenizer.vocab_size}")

In [ ]:
# Cell 4 – Keyword map
KEYWORD_MAP = {
    "ast__Import":       {"keyword": "import",   "forbidden_nodes": ["Import", "ImportFrom"]},
    "ast__ImportFrom":   {"keyword": "from",     "forbidden_nodes": ["Import", "ImportFrom"]},
    "ast__Break":        {"keyword": "break",    "forbidden_nodes": ["Break"]},
    "ast__Pass":         {"keyword": "pass",     "forbidden_nodes": ["Pass"]},
    "ast__Continue":     {"keyword": "continue", "forbidden_nodes": ["Continue"]},
    "ast__Assert":       {"keyword": "assert",   "forbidden_nodes": ["Assert"]},
    "ast__If":           {"keyword": "if",       "forbidden_nodes": ["If", "IfExp"]},
    "ast__While":        {"keyword": "while",    "forbidden_nodes": ["While"]},
    "ast__For":          {"keyword": "for",      "forbidden_nodes": ["For", "AsyncFor", "ListComp", "SetComp", "DictComp", "GeneratorExp"]},
    "ast__Return":       {"keyword": "return",   "forbidden_nodes": ["Return"]},
    "ast__With":         {"keyword": "with",     "forbidden_nodes": ["With", "AsyncWith"]},
    "ast__Raise":        {"keyword": "raise",    "forbidden_nodes": ["Raise"]},
    "ast__Delete":       {"keyword": "del",      "forbidden_nodes": ["Delete"]},
    "ast__Global":       {"keyword": "global",   "forbidden_nodes": ["Global"]},
    "ast__Nonlocal":     {"keyword": "nonlocal", "forbidden_nodes": ["Nonlocal"]},
    "ast__Lambda":       {"keyword": "lambda",   "forbidden_nodes": ["Lambda"]},
    "ast__Yield":        {"keyword": "yield",    "forbidden_nodes": ["Yield", "YieldFrom"]},
    "ast__YieldFrom":    {"keyword": "yield",    "forbidden_nodes": ["Yield", "YieldFrom"]},
    "ast__Try":          {"keyword": "try",      "forbidden_nodes": ["Try"]},
    "ast__ClassDef":     {"keyword": "class",    "forbidden_nodes": ["ClassDef"]},
    "ast__FunctionDef":  {"keyword": "def",      "forbidden_nodes": ["FunctionDef", "AsyncFunctionDef"]},
    "ast__AsyncFor":     {"keyword": "async",    "forbidden_nodes": ["AsyncFor", "AsyncFunctionDef", "AsyncWith"]},
    "ast__AsyncFunctionDef": {"keyword": "async", "forbidden_nodes": ["AsyncFor", "AsyncFunctionDef", "AsyncWith"]},
    "ast__AsyncWith":    {"keyword": "async",    "forbidden_nodes": ["AsyncFor", "AsyncFunctionDef", "AsyncWith"]},
    "builtin__list":     {"keyword": "list",     "forbidden_nodes": [],  "forbidden_names": ["list"]},
    "builtin__set":      {"keyword": "set",      "forbidden_nodes": [],  "forbidden_names": ["set"]},
    "builtin__map":      {"keyword": "map",      "forbidden_nodes": [],  "forbidden_names": ["map"]},
    "builtin__open":     {"keyword": "open",     "forbidden_nodes": [],  "forbidden_names": ["open"]},
    "builtin__type":     {"keyword": "type",     "forbidden_nodes": [],  "forbidden_names": ["type"]},
    "builtin__hash":     {"keyword": "hash",     "forbidden_nodes": [],  "forbidden_names": ["hash"]},
    "builtin__id":       {"keyword": "id",       "forbidden_nodes": [],  "forbidden_names": ["id"]},
    "builtin__all":      {"keyword": "all",      "forbidden_nodes": [],  "forbidden_names": ["all"]},
    "builtin__any":      {"keyword": "any",      "forbidden_nodes": [],  "forbidden_names": ["any"]},
    "builtin__sum":      {"keyword": "sum",      "forbidden_nodes": [],  "forbidden_names": ["sum"]},
    "builtin__min":      {"keyword": "min",      "forbidden_nodes": [],  "forbidden_names": ["min"]},
    "builtin__max":      {"keyword": "max",      "forbidden_nodes": [],  "forbidden_names": ["max"]},
    "builtin__next":     {"keyword": "next",     "forbidden_nodes": [],  "forbidden_names": ["next"]},
    "builtin__input":    {"keyword": "input",    "forbidden_nodes": [],  "forbidden_names": ["input"]},
    "builtin__len":      {"keyword": "len",      "forbidden_nodes": [],  "forbidden_names": ["len"]},
    "builtin__range":    {"keyword": "range",    "forbidden_nodes": [],  "forbidden_names": ["range"]},
    "builtin__filter":   {"keyword": "filter",   "forbidden_nodes": [],  "forbidden_names": ["filter"]},
    "builtin__print":    {"keyword": "print",    "forbidden_nodes": [],  "forbidden_names": ["print"]},
    "builtin__int":      {"keyword": "int",      "forbidden_nodes": [],  "forbidden_names": ["int"]},
    "builtin__float":    {"keyword": "float",    "forbidden_nodes": [],  "forbidden_names": ["float"]},
    "builtin__str":      {"keyword": "str",      "forbidden_nodes": [],  "forbidden_names": ["str"]},
    "builtin__bool":     {"keyword": "bool",     "forbidden_nodes": [],  "forbidden_names": ["bool"]},
    "builtin__round":    {"keyword": "round",    "forbidden_nodes": [],  "forbidden_names": ["round"]},
    "builtin__zip":      {"keyword": "zip",      "forbidden_nodes": [],  "forbidden_names": ["zip"]},
    "builtin__sorted":   {"keyword": "sorted",   "forbidden_nodes": [],  "forbidden_names": ["sorted"]},
    "builtin__super":    {"keyword": "super",    "forbidden_nodes": [],  "forbidden_names": ["super"]},
    "builtin__iter":     {"keyword": "iter",     "forbidden_nodes": [],  "forbidden_names": ["iter"]},
    "builtin__object":   {"keyword": "object",   "forbidden_nodes": [],  "forbidden_names": ["object"]},
    "builtin__bytes":    {"keyword": "bytes",    "forbidden_nodes": [],  "forbidden_names": ["bytes"]},
    "builtin__dict":     {"keyword": "dict",     "forbidden_nodes": [],  "forbidden_names": ["dict"]},
    "builtin__tuple":    {"keyword": "tuple",    "forbidden_nodes": [],  "forbidden_names": ["tuple"]},
    "builtin__property": {"keyword": "property", "forbidden_nodes": [],  "forbidden_names": ["property"]},
    "builtin__complex":  {"keyword": "complex",  "forbidden_nodes": [],  "forbidden_names": ["complex"]},
    "builtin__reversed": {"keyword": "reversed", "forbidden_nodes": [],  "forbidden_names": ["reversed"]},
}

print(f"KEYWORD_MAP: {len(KEYWORD_MAP)} objects")
print(f"  AST:     {sum(1 for k in KEYWORD_MAP if k.startswith('ast__'))}")
print(f"  Builtin: {sum(1 for k in KEYWORD_MAP if k.startswith('builtin__'))}")

In [ ]:
# Cell 5 – IDENTIFIER_VARIANTS definition (Section 2.3)
# Identifiers that contain the keyword as a substring but are NOT the keyword itself.
# Used for Category C prompts (variable/function names containing the keyword).

IDENTIFIER_VARIANTS = {
    "import":   ["important_data", "reimport_flag", "import_path_str"],
    "break":    ["breakdown_count", "breakpoint_flag", "unbreakable"],
    "pass":     ["passport_number", "password_hash", "bypass_mode"],
    "continue": ["continuous_mode", "discontinue_flag", "continuum"],
    "for":      ["format_string", "information", "formula_result", "before_update"],
    "if":       ["notification", "verification", "modification", "amplifier"],
    "while":    ["meanwhile_flag", "worthwhile", "awhile_counter"],
    "return":   ["return_value_str", "unreturnable", "return_label"],
    "assert":   ["assertive_mode", "reassert_count"],
    "with":     ["width_value", "withdraw_amount", "within_range"],
    "class":    ["classification", "classic_mode", "subclass_name"],
    "def":      ["default_value", "undefined_flag", "defiant_mode", "defense_level"],
    "yield":    ["yielded_count", "unyielding"],
    "try":      ["retry_count", "country_code", "entry_point"],
    "raise":    ["raised_flag", "fundraise_total", "praise_count"],
    "del":      ["delivery_date", "delta_value", "model_name", "delegate_to"],
    "global":   ["global_str_ref", "globalization"],
    "nonlocal": ["nonlocal_ref_str"],
    "lambda":   ["lambda_str_ref"],
    "async":    ["async_str_label"],
    # Builtins
    "list":     ["checklist_items", "listing_data", "blacklist_mode"],
    "set":      ["settings_config", "reset_flag", "offset_value", "sunset_time"],
    "map":      ["mapped_result", "roadmap_items", "bitmap_size"],
    "dict":     ["dictionary_words", "verdict_text", "dictation_mode"],
    "open":     ["opening_hours", "reopening", "openly_shared"],
    "type":     ["typewriter_mode", "archetype", "prototype_id"],
    "print":    ["printable_chars", "fingerprint_id", "blueprint_ref"],
    "int":      ["interval_ms", "painting_id", "intelligence", "pointer_val"],
    "float":    ["floating_label", "afloat_status"],
    "str":      ["string_data", "stream_id", "structure_type", "destroy_flag"],
    "range":    ["arrangement", "strange_value", "orange_count"],
    "sum":      ["summary_text", "consumer_id", "assumption"],
    "max":      ["maximum_str", "climax_point"],
    "min":      ["minimum_str", "admin_level", "minute_count", "dominion"],
    "len":      ["length_str", "lender_id", "silent_mode", "calendar_ref"],
    "hash":     ["hashtag_count", "rehash_data"],
    "bool":     ["boolean_str", "booleanize"],
    "round":    ["roundup_notes", "background_val", "surround_count"],
    "zip":      ["zipcode_val", "unzip_path"],
    "all":      ["alliance_name", "install_path", "wallpaper"],
    "any":      ["company_name", "many_items", "tyranny"],
    "next":     ["next_str_ref", "annex_id"],
    "input":    ["input_str_ref"],
    "filter":   ["filter_str_ref", "unfiltered_data"],
    "object":   ["objection_text", "objective_id"],
    "id":       ["idea_count", "identity_str", "video_ref", "oxide_level"],
    "super":    ["supervisor_name", "superfluous"],
    "iter":     ["literal_count", "iterator_str_ref"],
    "bytes":    ["bytes_str_ref", "gigabytes_val"],
    "sorted":   ["sorted_str_ref", "unsorted_data"],
    "tuple":    ["tuple_str_ref", "quintuple_val"],
    "property": ["property_str_ref"],
    "complex":  ["complexity_score", "complex_str_ref"],
    "reversed": ["reversed_str_ref"],
}

print(f"IDENTIFIER_VARIANTS: {len(IDENTIFIER_VARIANTS)} keywords covered")

In [ ]:
# Cell 6 – Word banks (Section 2.3)

VAR_NAMES = [
    "data", "result", "value", "item", "record", "entry", "output",
    "signal", "status", "config", "mode", "level", "target", "source",
    "handler", "token", "state", "buffer", "counter", "index",
    "offset", "block", "phase", "cycle", "frame", "chunk", "batch",
    "queue", "score", "label", "flag", "limit", "depth", "width_val",
]

STRING_WORDS = [
    "waiting", "ready", "starting", "completed", "processing",
    "running", "loading", "checking", "updating", "building",
    "reading", "writing", "sending", "receiving", "tracking",
    "reference", "documentation", "deployment", "production",
    "analysis", "operation", "execution", "validation", "review",
    "schedule", "planning", "progress", "workflow", "pipeline",
    "resource", "component", "structure", "algorithm", "protocol",
    "standard", "baseline", "threshold", "interval", "duration",
    "capacity", "frequency", "priority", "sequence", "pattern",
    "segment", "fragment", "section", "portion", "division",
]

CONTEXT_WRAPPERS = [
    "{snippet}",
    "x = 1\n{snippet}\ny = 2",
    "data = []\n{snippet}\nresult = None",
    "def func():\n    {snippet}\n    return None",
    "def process():\n    {snippet}",
    "class Obj:\n    def method(self):\n        {snippet}",
]

all_keywords = set(v["keyword"] for v in KEYWORD_MAP.values())
safe_vars = [v for v in VAR_NAMES if not any(kw in v for kw in all_keywords)]
print(f"VAR_NAMES: {len(VAR_NAMES)} -> {len(safe_vars)} after keyword filtering")
removed = set(VAR_NAMES) - set(safe_vars)
if removed:
    print(f"  Removed: {removed}")
VAR_NAMES = safe_vars

In [ ]:
# Cell 7 – Validation functions (Section 3)

def get_all_ast_info(code_string):
    try:
        tree = ast_module.parse(code_string)
    except SyntaxError:
        return None, None
    node_types = set()
    call_names = set()
    for node in ast_module.walk(tree):
        node_types.add(type(node).__name__)
        if isinstance(node, ast_module.Name):
            pass
        if isinstance(node, ast_module.Call):
            if isinstance(node.func, ast_module.Name):
                call_names.add(node.func.id)
    return node_types, call_names

def check_concept_absent(code_string, obj_key):
    info = KEYWORD_MAP[obj_key]
    node_types, call_names = get_all_ast_info(code_string)
    if node_types is None:
        return False
    for forbidden in info["forbidden_nodes"]:
        if forbidden in node_types:
            return False
    if "forbidden_names" in info:
        for forbidden_name in info["forbidden_names"]:
            if forbidden_name in call_names:
                return False
    return True

def check_token_present(code_string, keyword, tokenizer):
    token_ids = tokenizer.encode(code_string, add_special_tokens=False)
    keyword_token_ids = tokenizer.encode(keyword, add_special_tokens=False)
    ids_list = token_ids
    kw_len = len(keyword_token_ids)
    for i in range(len(ids_list) - kw_len + 1):
        if ids_list[i:i+kw_len] == keyword_token_ids:
            return True
    for tid in token_ids:
        decoded = tokenizer.decode([tid]).strip()
        if decoded == keyword:
            return True
    return False

def validate_prompt(code_string, obj_key):
    keyword = KEYWORD_MAP[obj_key]["keyword"]
    node_types, call_names = get_all_ast_info(code_string)
    if node_types is None:
        return False
    if not check_concept_absent(code_string, obj_key):
        return False
    if not check_token_present(code_string, keyword, tokenizer):
        return False
    return True

print("Validation functions defined.")

In [ ]:
# Cell 8 – Generation functions + main loop
from tqdm.auto import tqdm

def generate_raw_prompts(keyword, n_per_category=10):
    candidates = []
    for _ in range(n_per_category):
        w1, w2, w3 = random.sample(STRING_WORDS, 3)
        v1, v2 = random.sample(VAR_NAMES, 2)
        wrapper = random.choice(CONTEXT_WRAPPERS)

        snippet_a = random.choice([
            f'{v1} = "waiting {keyword} {w1}"',
            f'{v1} = "{w1} {keyword} {w2} reference"',
            f'{v1} = "the {keyword} of {w1}"',
            f'{v1} = "no {keyword} needed for {w1}"',
            f'{v1} = "{w1} {keyword} {w2}"',
        ])
        candidates.append((wrapper.replace("{snippet}", snippet_a), "A_string"))

        snippet_b = random.choice([
            f'{v1} = 42  # {keyword} {w1} see docs',
            f'# {keyword} {w1} {w2} reference\n{v1} = 10',
            f'{v1} = []  # TODO {keyword} this later',
            f'# note: {keyword} not needed for {w1}\n{v1} = True',
        ])
        candidates.append((wrapper.replace("{snippet}", snippet_b), "B_comment"))

        ident_variants = IDENTIFIER_VARIANTS.get(keyword, [])
        if ident_variants:
            ident = random.choice(ident_variants)
            snippet_c = random.choice([
                f'{ident} = 42',
                f'{ident} = "{w1}"',
                f'{ident} = True',
            ])
            candidates.append((wrapper.replace("{snippet}", snippet_c), "C_identifier"))

        snippet_d = random.choice([
            f'{v1} = {{"{keyword}": True, "{w1}": 42}}',
            f'{v1} = {{"{w1}": 10, "mode": "{keyword}"}}',
            f'{v1} = {{"action": "{w1}", "{keyword}": "{w2}"}}',
        ])
        candidates.append((wrapper.replace("{snippet}", snippet_d), "D_dictkey"))

        snippet_e = random.choice([
            f'print("{w1} {keyword} {w2}")',
            f'print("starting {keyword} {w1}")',
            f'print("{keyword} completed for {w1}")',
        ])
        candidates.append((wrapper.replace("{snippet}", snippet_e), "E_print"))

    return candidates

def generate_for_object(obj_key, target_n=N_TARGET):
    keyword = KEYWORD_MAP[obj_key]["keyword"]
    raw = generate_raw_prompts(keyword, n_per_category=target_n * OVERSHOOT_FACTOR // 5)
    valid = [(t, c) for t, c in raw if validate_prompt(t, obj_key)]

    by_cat = {}
    for text, cat in valid:
        by_cat.setdefault(cat, []).append(text)

    per_cat = target_n // 5
    selected = []
    for cat in ["A_string", "B_comment", "C_identifier", "D_dictkey", "E_print"]:
        selected.extend(by_cat.get(cat, [])[:per_cat])

    if len(selected) < target_n:
        remaining = [t for t, c in valid if t not in selected]
        selected.extend(remaining[:target_n - len(selected)])

    return selected[:target_n]

# Main generation loop
all_prompts = {}
for obj_key in tqdm(sorted(KEYWORD_MAP.keys()), desc="Generating"):
    prompts = generate_for_object(obj_key)
    all_prompts[obj_key] = prompts
    status = "OK" if len(prompts) >= N_TARGET else f"SHORT ({len(prompts)})"
    print(f"  {obj_key}: {len(prompts)} valid [{status}]")

total = sum(len(v) for v in all_prompts.values())
short = [k for k, v in all_prompts.items() if len(v) < N_TARGET]
print(f"\nTotal: {total} prompts across {len(all_prompts)} objects")
if short:
    print(f"WARNING: {len(short)} objects have < {N_TARGET} prompts: {short}")

In [ ]:
# Cell 9 – Save checker prompts
records = []
for obj_key, prompts in sorted(all_prompts.items()):
    for i, text in enumerate(prompts):
        records.append({
            "object": obj_key,
            "keyword": KEYWORD_MAP[obj_key]["keyword"],
            "variation_id": i,
            "prompt_text": text,
        })

out_path = os.path.join(DATA_DIR, OUTPUT_FILE)
pd.DataFrame(records).to_parquet(out_path, index=False)
print(f"Saved: {out_path}")
print(f"Total rows: {len(records)}")